# Theoretical Extensions — DCF

Analytical companion to the empirical evaluation notebooks. No dataset downloads or GPU required.

Covers:
1. **Lemma 1 — Lipschitz Consistency Bound:** for a classifier with Lipschitz constant L and decision margin m, Label Stability satisfies S ≥ 1 − L·ε/m
2. **Lemma 2 — CAG Dichotomy:** empirical proof across 16 configurations that models fall into Overconfident (CAG > 0) or Consistently Wrong (CAG < 0) — no well-calibrated region found
3. **Architecture scaling analysis:** CS vs model depth and parameter count
4. **Medical case study:** OCTMNIST — how CAG reveals dangerous failure modes that accuracy hides
5. **Master results table:** all 16 CNN configurations + perturbation analysis + LaTeX export

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
OUTPUT_DIR = r'E:\decision_consistency_analysis'

import os, json, csv
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

os.makedirs(os.path.join(OUTPUT_DIR, 'figures'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'tables'),  exist_ok=True)
print('Output:', OUTPUT_DIR)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# LEMMA 1 — Lipschitz Consistency Bound
# For a classifier f with Lipschitz constant L and decision margin m,
# Label Stability satisfies: S >= 1 - (L * eps) / m
# ─────────────────────────────────────────────────────────────────────────────

def lipschitz_min_cs(L, eps, margin):
    flip_prob = min(1.0, (L * eps) / margin)
    return 1 - flip_prob, flip_prob

eps    = 0.01
margin = 0.1
L_vals = [1.0, 2.0, 5.0, 10.0]

print('Lemma 1 — Lipschitz Consistency Bound')
print(f'  ε = {eps},  margin = {margin}')
print(f'{"L":>6} {"Min CS":>10} {"Max Flip":>10}')
print('-' * 30)
for L in L_vals:
    min_cs, max_flip = lipschitz_min_cs(L, eps, margin)
    print(f'{L:>6.1f} {min_cs:>10.4f} {max_flip:>10.4f}')

L_range = np.linspace(0.5, 20, 200)
cs_bounds = [lipschitz_min_cs(L, eps, margin)[0] for L in L_range]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(L_range, cs_bounds, linewidth=2, color='steelblue')
ax.fill_between(L_range, cs_bounds, 1, alpha=0.15, color='steelblue')
ax.axhline(0.8, color='tomato', linestyle='--', lw=1.5, label='Typical CS threshold (0.8)')
ax.set_xlabel('Lipschitz Constant L', fontsize=11)
ax.set_ylabel('Minimum Guaranteed Consistency Score', fontsize=11)
ax.set_title('Lemma 1: Lower Bound on CS as a Function of L\n'
             r'$S \geq 1 - L\varepsilon / m$', fontsize=11)
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
path = os.path.join(OUTPUT_DIR, 'figures', 'lemma1_lipschitz_bound.png')
plt.savefig(path, dpi=300, bbox_inches='tight'); plt.close()
print('Saved:', path)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Empirical results — all 16 CNN configurations
# ─────────────────────────────────────────────────────────────────────────────

all_results = [
    # (dataset, model, accuracy, CS, CAG)
    ('CIFAR-10',    'ResNet-18',   0.876, 0.7822,  0.0938),
    ('CIFAR-10',    'ResNet-50',   0.865, 0.7889,  0.0761),
    ('CIFAR-10',    'VGG-16',      0.856, 0.7767,  0.0793),
    ('CIFAR-10',    'MobileNetV2', 0.874, 0.7757,  0.0983),
    ('STL-10',      'ResNet-18',   0.939, 0.8571,  0.0819),
    ('STL-10',      'ResNet-50',   0.969, 0.8730,  0.0960),
    ('STL-10',      'VGG-16',      0.936, 0.8383,  0.0977),
    ('STL-10',      'MobileNetV2', 0.935, 0.8538,  0.0812),
    ('Intel-Scene', 'ResNet-18',   0.574, 0.8659, -0.2919),
    ('Intel-Scene', 'ResNet-50',   0.526, 0.8478, -0.3218),
    ('Intel-Scene', 'VGG-16',      0.742, 0.8484, -0.1064),
    ('Intel-Scene', 'MobileNetV2', 0.599, 0.8388, -0.2398),
    ('OCTMNIST',    'ResNet-18',   0.470, 0.8612, -0.3912),
    ('OCTMNIST',    'ResNet-50',   0.526, 0.8466, -0.3206),
    ('OCTMNIST',    'VGG-16',      0.628, 0.8303, -0.2023),
    ('OCTMNIST',    'MobileNetV2', 0.513, 0.8146, -0.3016),
]

print(f'{"Dataset":<14} {"Model":<14} {"Acc":>6} {"CS":>7} {"CAG":>8} {"Mode":<20}')
print('-' * 75)
for ds, model, acc, cs, cag in all_results:
    mode = 'Overconfident' if cag > 0 else 'Consistently Wrong'
    print(f'{ds:<14} {model:<14} {acc:>6.3f} {cs:>7.4f} {cag:>8.4f} {mode:<20}')

overconf  = sum(1 for *_, cag in all_results if cag > 0)
cons_wrong = sum(1 for *_, cag in all_results if cag < 0)
print(f'\nOverconfident: {overconf}  |  Consistently Wrong: {cons_wrong}  |  Well-Calibrated: 0')
print('→ Lemma 2 (CAG Dichotomy): no configuration falls in the well-calibrated region.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# LEMMA 2 — CAG Dichotomy visualisation
# ─────────────────────────────────────────────────────────────────────────────

configs = [f'{ds[:4]}-{model[:8]}' for ds, model, *_ in all_results]
cag_vals = [cag for *_, cag in all_results]
colors   = ['tomato' if c > 0 else 'steelblue' for c in cag_vals]

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(configs, cag_vals, color=colors)
ax.axvline(0, color='black', linewidth=2)
ax.set_xlabel('CAG  (Accuracy − Consistency Score)', fontsize=11)
ax.set_title('Lemma 2: CAG Dichotomy — All 16 Configurations\n'
             'Red = Overconfident (CAG > 0)   Blue = Consistently Wrong (CAG < 0)', fontsize=11)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
path = os.path.join(OUTPUT_DIR, 'figures', 'lemma2_cag_dichotomy.png')
plt.savefig(path, dpi=300, bbox_inches='tight'); plt.close()
print('Saved:', path)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Architecture scaling: CS vs model depth and parameter count
# ─────────────────────────────────────────────────────────────────────────────

arch_info = {
    'ResNet-18':   {'depth': 18,  'params_m': 11.7},
    'ResNet-50':   {'depth': 50,  'params_m': 25.6},
    'VGG-16':      {'depth': 16,  'params_m': 138.4},
    'MobileNetV2': {'depth': 53,  'params_m': 3.5},
}
datasets_plot = ['CIFAR-10', 'STL-10']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ds in datasets_plot:
    ds_rows = [(model, acc, cs, cag) for d, model, acc, cs, cag in all_results if d == ds]
    depths  = [arch_info[m]['depth']   for m, *_ in ds_rows]
    cs_vals = [cs                       for _, _, cs, _ in ds_rows]
    axes[0].scatter(depths, cs_vals, s=100, label=ds)

axes[0].set_xlabel('Model Depth (layers)', fontsize=11)
axes[0].set_ylabel('Consistency Score', fontsize=11)
axes[0].set_title('CS vs Depth', fontsize=11)
axes[0].legend(); axes[0].grid(alpha=0.3)

for ds in datasets_plot:
    ds_rows = [(model, acc, cs, cag) for d, model, acc, cs, cag in all_results if d == ds]
    params  = [arch_info[m]['params_m'] for m, *_ in ds_rows]
    cs_vals = [cs                        for _, _, cs, _ in ds_rows]
    axes[1].scatter(params, cs_vals, s=100, label=ds)
    for (model, *_, cs), p in zip(ds_rows, params):
        axes[1].annotate(model[:8], (p, cs), fontsize=7, ha='center', va='bottom')

axes[1].set_xlabel('Parameters (millions)', fontsize=11)
axes[1].set_xscale('log')
axes[1].set_ylabel('Consistency Score', fontsize=11)
axes[1].set_title('Efficiency–Consistency Trade-off', fontsize=11)
axes[1].legend(); axes[1].grid(alpha=0.3)

fig.suptitle('Architecture Scaling Analysis', fontsize=13)
plt.tight_layout()
path = os.path.join(OUTPUT_DIR, 'figures', 'architecture_scaling.png')
plt.savefig(path, dpi=300, bbox_inches='tight'); plt.close()
print('Saved:', path)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Medical case study — OCTMNIST
# Shows that a 47% accuracy on retinal OCT classification sounds merely
# 'needs improvement', but CAG = −0.39 reveals it is CONSISTENTLY WRONG —
# the same misclassification happens reliably, which is dangerous in deployment.
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

ax = axes[0]
ax.bar(['Accuracy\n(Standard)', 'Consistency\n(DCF)'], [0.470, 0.861],
       color=['tomato', 'steelblue'], edgecolor='white')
ax.set_ylim(0, 1)
ax.set_title('OCTMNIST / ResNet-18\nStandard vs DCF evaluation', fontsize=10)
ax.set_ylabel('Score')
for x, v in enumerate([0.470, 0.861]):
    ax.text(x, v + 0.02, f'{v:.3f}', ha='center', fontsize=11)

ax = axes[1]
ax.bar(['CAG'], [-0.3912], color='steelblue', edgecolor='white')
ax.axhline(0, color='black', linewidth=2)
ax.set_ylim(-0.6, 0.2)
ax.set_title('CAG = −0.39\nConsistently Wrong regime', fontsize=10)
ax.set_ylabel('CAG  (Accuracy − CS)')
ax.text(0, -0.42, 'CAG = −0.391', ha='center', fontsize=11, fontweight='bold', color='steelblue')
ax.text(0, -0.52, 'CONSISTENTLY WRONG', ha='center', fontsize=10, color='tomato', fontweight='bold')

fig.suptitle('Medical Case Study: How DCF Reveals Deployment Risk\n'
             'Accuracy (47%) looks tolerable — CAG (−0.39) flags systematic bias', fontsize=11)
plt.tight_layout()
path = os.path.join(OUTPUT_DIR, 'figures', 'medical_case_study.png')
plt.savefig(path, dpi=300, bbox_inches='tight'); plt.close()
print('Saved:', path)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Master results table — CSV + LaTeX
# ─────────────────────────────────────────────────────────────────────────────

import pandas as pd

df = pd.DataFrame(all_results,
                  columns=['Dataset','Model','Accuracy','CS','CAG'])
df['CAG_Mode'] = df['CAG'].apply(lambda x: 'Overconfident' if x > 0 else 'Consistently Wrong')
df['Params_M'] = df['Model'].map({k: v['params_m'] for k, v in arch_info.items()})
df = df.sort_values('CS', ascending=False).reset_index(drop=True)

csv_path = os.path.join(OUTPUT_DIR, 'tables', 'master_results.csv')
df.to_csv(csv_path, index=False)
print('CSV saved:', csv_path)

# LaTeX version
latex_lines = [
    r'\begin{table}[htbp]',
    r'\centering',
    r'\caption{Complete CNN Results — Decision Consistency Framework}',
    r'\label{tab:master_results}',
    r'\begin{tabular}{llcccl}',
    r'\toprule',
    r'\textbf{Dataset} & \textbf{Model} & \textbf{Accuracy} & \textbf{CS} & \textbf{CAG} & \textbf{Mode} \\',
    r'\midrule',
]
for _, row in df.iterrows():
    latex_lines.append(
        f"{row['Dataset']} & {row['Model']} & {row['Accuracy']:.3f} & "
        f"{row['CS']:.4f} & {row['CAG']:.4f} & {row['CAG_Mode']} \\\\"
    )
latex_lines += [r'\bottomrule', r'\end{tabular}', r'\end{table}']

tex_path = os.path.join(OUTPUT_DIR, 'tables', 'master_results.tex')
with open(tex_path, 'w') as f: f.write('\n'.join(latex_lines))
print('LaTeX saved:', tex_path)

print('\nTop 5 configurations by Consistency Score:')
print(df[['Dataset','Model','CS','CAG','CAG_Mode']].head(5).to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Perturbation analysis — flip rates across datasets
# ─────────────────────────────────────────────────────────────────────────────

perturbation_data = [
    {'Dataset':'CIFAR-10',    'Perturbation':'jpeg',       'Flip_Rate':0.640, 'Impact':'Very High'},
    {'Dataset':'CIFAR-10',    'Perturbation':'blur',       'Flip_Rate':0.370, 'Impact':'High'},
    {'Dataset':'CIFAR-10',    'Perturbation':'noise',      'Flip_Rate':0.357, 'Impact':'High'},
    {'Dataset':'CIFAR-10',    'Perturbation':'rotation',   'Flip_Rate':0.327, 'Impact':'High'},
    {'Dataset':'CIFAR-10',    'Perturbation':'brightness', 'Flip_Rate':0.140, 'Impact':'Medium'},
    {'Dataset':'CIFAR-10',    'Perturbation':'contrast',   'Flip_Rate':0.097, 'Impact':'Low'},
    {'Dataset':'STL-10',      'Perturbation':'jpeg',       'Flip_Rate':0.366, 'Impact':'High'},
    {'Dataset':'STL-10',      'Perturbation':'rotation',   'Flip_Rate':0.260, 'Impact':'Medium'},
    {'Dataset':'STL-10',      'Perturbation':'blur',       'Flip_Rate':0.180, 'Impact':'Medium'},
    {'Dataset':'STL-10',      'Perturbation':'noise',      'Flip_Rate':0.110, 'Impact':'Low'},
    {'Dataset':'Intel-Scene', 'Perturbation':'jpeg',       'Flip_Rate':0.553, 'Impact':'Very High'},
    {'Dataset':'Intel-Scene', 'Perturbation':'rotation',   'Flip_Rate':0.297, 'Impact':'High'},
    {'Dataset':'OCTMNIST',    'Perturbation':'noise',      'Flip_Rate':0.725, 'Impact':'Very High'},
    {'Dataset':'OCTMNIST',    'Perturbation':'jpeg',       'Flip_Rate':0.450, 'Impact':'High'},
    {'Dataset':'OCTMNIST',    'Perturbation':'blur',       'Flip_Rate':0.300, 'Impact':'Medium'},
]

pert_df = pd.DataFrame(perturbation_data)
pert_csv = os.path.join(OUTPUT_DIR, 'tables', 'perturbation_flip_rates.csv')
pert_df.to_csv(pert_csv, index=False)
print('Perturbation table saved:', pert_csv)

# Bar chart: most disruptive perturbation per dataset
fig, axes = plt.subplots(1, 4, figsize=(14, 4), sharey=True)
for ax, ds in zip(axes, ['CIFAR-10','STL-10','Intel-Scene','OCTMNIST']):
    sub = pert_df[pert_df['Dataset'] == ds].sort_values('Flip_Rate', ascending=False)
    ax.bar(sub['Perturbation'], sub['Flip_Rate'], color='darkorange', edgecolor='white')
    ax.set_title(ds, fontsize=10)
    ax.set_ylim(0, 0.8)
    ax.set_xticklabels(sub['Perturbation'], rotation=45, ha='right', fontsize=8)
    ax.grid(axis='y', alpha=0.3)
axes[0].set_ylabel('Label Flip Rate')
fig.suptitle('Per-Perturbation Label Flip Rate by Dataset', fontsize=12)
plt.tight_layout()
path = os.path.join(OUTPUT_DIR, 'figures', 'perturbation_flip_rates.png')
plt.savefig(path, dpi=300, bbox_inches='tight'); plt.close()
print('Saved:', path)